# **Import**

In [1]:
import pandas as pd
import numpy as np

from tqdm import tqdm

from sklearn.metrics import mean_absolute_error, mean_squared_error

from lightgbm import LGBMRegressor

# **Data Load**

In [2]:
cd /content/drive/MyDrive/[Projects]/Dacon/제3회 국민대학교 AI빅데이터 분석 경진대회/Data

/content/drive/MyDrive/[Projects]/Dacon/제3회 국민대학교 AI빅데이터 분석 경진대회/Data


In [3]:
train_df = pd.read_csv('./train.csv')
pair_df = pd.read_csv('./pairs.csv')

In [54]:
train_df

,item_id,year,month,seq,type,hs4,weight,quantity,value
0,DEWLVASR,2022,1,1.0,1,3038,14858.0,0.0,32688.0
1,ELQGMQWE,2022,1,1.0,1,2002,62195.0,0.0,110617.0
2,AHMDUILJ,2022,1,1.0,1,2102,18426.0,0.0,72766.0
3,XIPPENFQ,2022,1,1.0,1,2501,20426.0,0.0,11172.0
4,FTSVTTSR,2022,1,1.0,1,2529,248000.0,0.0,143004.0
...,...,...,...,...,...,...,...,...,...
10831,XIFHSOWQ,2025,7,3.0,1,8708,352.0,0.0,12937.0
10832,FITUEHWN,2025,7,3.0,1,8714,655.0,900.0,16054.0
10833,UGEQLMXM,2025,7,3.0,1,8714,758.0,0.0,74377.0
10834,BLANHGYY,2025,7,3.0,1,9022,345.0,2.0,69720.0


In [55]:
pair_df

,leading_item_id,following_item_id,best_lag,max_corr
0,AANGBULD,APQGTRMF,5,-0.443984
1,AANGBULD,DEWLVASR,6,0.640221
2,AANGBULD,DNMPSKTB,4,-0.410635
3,AANGBULD,EVBVXETX,6,0.436623
4,AANGBULD,FTSVTTSR,3,0.531400
...,...,...,...,...
1420,ZXERAXWP,DBWLZWNK,6,-0.470150
1421,ZXERAXWP,FITUEHWN,4,0.406369
1422,ZXERAXWP,MIRCVAMV,3,-0.435495
1423,ZXERAXWP,UIFPPCLR,1,-0.526205


# **Build Model per Items**

In [4]:
monthly = (
    train_df
    .groupby(['item_id', 'year', 'month'], as_index=False)['value']
    .sum()
)
monthly['ym'] = pd.to_datetime(monthly['year'].astype(str) + '-' + monthly['month'].astype(str))
pivot = monthly.pivot(index='item_id', columns='ym', values='value')
pivot = pivot.fillna(0)
pivot

ym,2022-01-01,2022-02-01,2022-03-01,2022-04-01,2022-05-01,2022-06-01,2022-07-01,2022-08-01,2022-09-01,2022-10-01,...,2024-10-01,2024-11-01,2024-12-01,2025-01-01,2025-02-01,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01
item_id,,,,,,,,,,,,,,,,,,,,,
AANGBULD,14276.0,52347.0,53549.0,0.0,26997.0,84489.0,0.0,0.0,0.0,0.0,...,428725.0,144248.0,26507.0,25691.0,25805.0,0.0,38441.0,0.0,441275.0,533478.0
AHMDUILJ,242705.0,120847.0,197317.0,126142.0,71730.0,149138.0,186617.0,169995.0,140547.0,89292.0,...,123085.0,143451.0,78649.0,125098.0,80404.0,157401.0,115509.0,127473.0,89479.0,101317.0
ANWUJOKX,0.0,0.0,0.0,63580.0,81670.0,26424.0,8470.0,0.0,0.0,80475.0,...,0.0,0.0,0.0,27980.0,0.0,0.0,0.0,0.0,0.0,0.0
APQGTRMF,383999.0,512813.0,217064.0,470398.0,539873.0,582317.0,759980.0,216019.0,537693.0,205326.0,...,683581.0,2147.0,0.0,25013.0,77.0,20741.0,2403.0,3543.0,32430.0,40608.0
ATLDMDBO,143097177.0,103568323.0,118403737.0,121873741.0,115024617.0,65716075.0,146216818.0,97552978.0,72341427.0,87454167.0,...,60276050.0,30160198.0,42613728.0,64451013.0,38667429.0,29354408.0,42450439.0,37136720.0,32181798.0,57090235.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YSYHGLQK,0.0,543.0,766.0,1108.0,859.0,1426.0,2413.0,638.0,0.0,1199.0,...,188.0,541.0,696.0,8710.0,3175.0,2624.0,0.0,182.0,2128.0,10651.0
ZCELVYQU,373859.0,59900.0,31158.0,594407.0,648232.0,496737.0,210179.0,0.0,70748.0,15512.0,...,0.0,609803.0,23712.0,654630.0,4496.0,1177300.0,1187539.0,26434.0,115631.0,270262.0
ZGJXVMNI,1154724.0,1337622.0,1662893.0,1561647.0,1603223.0,1641942.0,1815161.0,1546959.0,1536799.0,1496906.0,...,3168505.0,3059865.0,1579976.0,1413293.0,3038078.0,2915914.0,3565526.0,3020051.0,2412781.0,2458481.0


In [29]:
max_self_lag = 3
ma_windows = [3, 6]
std_windows = [3, 6]

def make_features_by_items(pair_df, leader, follower):
    all_rows = []  # 함수 안에서 초기화

    row = pair_df[(pair_df['leading_item_id']==leader) &
                  (pair_df['following_item_id']==follower)].iloc[0]
    best_lag = int(row['best_lag'])

    s_lead = pivot.loc[leader]
    s_follow = pivot.loc[follower]

    for ym in pivot.columns:
        rec = {'month': ym, 'leader_item': leader, 'follower_item': follower}

        # Leader lag 주변 추가
        for lag_shift in [-1, 0, 1]:
            lag = best_lag + lag_shift
            if lag <= 0:
                continue
            rec[f'leader_lag{lag}'] = s_lead.get(ym - pd.DateOffset(months=lag), np.nan)

        # Follower lag
        for l in range(1, max_self_lag + 1):
            rec[f'follower_lag{l}'] = s_follow.get(ym - pd.DateOffset(months=l), np.nan)

        # 이동평균
        for w in ma_windows:
            rec[f'leader_ma{w}'] = s_lead[:ym - pd.DateOffset(months=1)].rolling(w).mean().iloc[-1] \
                if len(s_lead[:ym - pd.DateOffset(months=1)]) >= w else np.nan
            rec[f'follower_ma{w}'] = s_follow[:ym - pd.DateOffset(months=1)].rolling(w).mean().iloc[-1] \
                if len(s_follow[:ym - pd.DateOffset(months=1)]) >= w else np.nan

        # STD Feature
        for w in std_windows:
            rec[f'leader_std{w}'] = s_lead[:ym - pd.DateOffset(months=1)].rolling(w).std().iloc[-1] \
                if len(s_lead[:ym - pd.DateOffset(months=1)]) >= w else np.nan
            rec[f'follower_std{w}'] = s_follow[:ym - pd.DateOffset(months=1)].rolling(w).std().iloc[-1] \
                if len(s_follow[:ym - pd.DateOffset(months=1)]) >= w else np.nan

        # DIFF Feature
        for l in range(1, max_self_lag):
            # Leader diff
            if not pd.isna(rec.get(f'leader_lag{l}')) and not pd.isna(rec.get(f'leader_lag{l+1}')):
                rec[f'leader_diff{l}'] = rec[f'leader_lag{l}'] - rec[f'leader_lag{l+1}']
            # Follower diff
            if not pd.isna(rec.get(f'follower_lag{l}')) and not pd.isna(rec.get(f'follower_lag{l+1}')):
                rec[f'follower_diff{l}'] = rec[f'follower_lag{l}'] - rec[f'follower_lag{l+1}']

        # Month number
        rec['month_num'] = ym.month

        # Target
        rec['target'] = s_follow.get(ym, np.nan)

        all_rows.append(rec)

    feature_df = pd.DataFrame(all_rows)
    feature_df = feature_df.sort_values(['leader_item', 'follower_item', 'month']).reset_index(drop=True)
    return feature_df

In [30]:
unique_pair = pair_df[['leading_item_id', 'following_item_id']].value_counts().reset_index().sort_values(['leading_item_id', 'following_item_id'], ascending=True).reset_index(drop=True)
unique_pair = unique_pair.drop(columns='count')
unique_pair.values

array([['AANGBULD', 'APQGTRMF'],
       ['AANGBULD', 'DEWLVASR'],
       ['AANGBULD', 'DNMPSKTB'],
       ...,
       ['ZXERAXWP', 'MIRCVAMV'],
       ['ZXERAXWP', 'UIFPPCLR'],
       ['ZXERAXWP', 'WHPUAOID']], dtype=object)

In [34]:
results_each_pairs = []
for leader, follower in tqdm(unique_pair.values, desc="Processing item pairs"):
    item_feature_df = make_features_by_items(pair_df, leader, follower)

    train_cutoff = pd.Timestamp('2025-07-01') - pd.DateOffset(months=1)
    val_month = pd.Timestamp('2025-07-01')

    train_df_features = item_feature_df[item_feature_df['month'] <= train_cutoff].copy()
    val_df_features = item_feature_df[item_feature_df['month'] == val_month].copy()

    x_cols = [c for c in train_df_features.columns if c not in ['month','leader_item','follower_item','target']]

    train_x = train_df_features[x_cols]
    train_y = train_df_features['target']
    val_x = val_df_features[x_cols]
    val_y = val_df_features['target']

    model = LGBMRegressor(n_estimators=1000, learning_rate=0.05, verbose=-1, random_state=42)
    model.fit(train_x, train_y, eval_set=[(val_x, val_y)], eval_metric='mae')

    pred_y = model.predict(val_x)

    results_each_pairs.append({
        'leader_item_id': leader,
        'follower_item_id': follower,
        'MAE': mean_absolute_error(val_y, pred_y),
        'RMSE': mean_squared_error(val_y, pred_y)**0.5
    })

Processing item pairs: 100%|██████████| 1425/1425 [06:35<00:00,  3.60it/s]


In [35]:
result_df = pd.DataFrame(results_each_pairs)
print(result_df['MAE'].mean(), result_df['RMSE'].mean())
result_df

1330752.8288798432 1330752.8288798432


,leader_item_id,follower_item_id,MAE,RMSE
0,AANGBULD,APQGTRMF,2.284762e+05,2.284762e+05
1,AANGBULD,DEWLVASR,1.085864e+05,1.085864e+05
2,AANGBULD,DNMPSKTB,1.523772e+06,1.523772e+06
3,AANGBULD,EVBVXETX,4.615490e+04,4.615490e+04
4,AANGBULD,FTSVTTSR,1.128071e+05,1.128071e+05
...,...,...,...,...
1420,ZXERAXWP,DBWLZWNK,4.226902e+04,4.226902e+04
1421,ZXERAXWP,FITUEHWN,5.823037e+04,5.823037e+04
1422,ZXERAXWP,MIRCVAMV,4.811885e+04,4.811885e+04
1423,ZXERAXWP,UIFPPCLR,2.606890e+04,2.606890e+04


In [37]:
result_df['MAE'].max(), result_df['MAE'].min()

(32213489.284084495, 1.7361022466493523)

# **Prediction**

In [39]:
def make_pred_features_by_items(pair_df, leader, follower):
    pred_month = pd.Timestamp('2025-08-01')
    all_rows = []

    row = pair_df[(pair_df['leading_item_id']==leader) &
                  (pair_df['following_item_id']==follower)].iloc[0]
    best_lag = int(row['best_lag'])

    s_lead = pivot.loc[leader]
    s_follow = pivot.loc[follower]

    rec = {'month': pred_month, 'leader_item': leader, 'follower_item': follower}

    # Leader lag 주변
    for lag_shift in [-1, 0, 1]:
        lag = best_lag + lag_shift
        if lag <= 0:
            continue
        rec[f'leader_lag{lag}'] = s_lead.get(pred_month - pd.DateOffset(months=lag), np.nan)

    # Follower lag
    for l in range(1, max_self_lag + 1):
        rec[f'follower_lag{l}'] = s_follow.get(pred_month - pd.DateOffset(months=l), np.nan)

    # 이동평균
    for w in ma_windows:
        rec[f'leader_ma{w}'] = s_lead[:pred_month - pd.DateOffset(months=1)].rolling(w).mean().iloc[-1] \
            if len(s_lead[:pred_month - pd.DateOffset(months=1)]) >= w else np.nan
        rec[f'follower_ma{w}'] = s_follow[:pred_month - pd.DateOffset(months=1)].rolling(w).mean().iloc[-1] \
            if len(s_follow[:pred_month - pd.DateOffset(months=1)]) >= w else np.nan

    # STD Feature
    for w in std_windows:
        rec[f'leader_std{w}'] = s_lead[:pred_month - pd.DateOffset(months=1)].rolling(w).std().iloc[-1] \
            if len(s_lead[:pred_month - pd.DateOffset(months=1)]) >= w else np.nan
        rec[f'follower_std{w}'] = s_follow[:pred_month - pd.DateOffset(months=1)].rolling(w).std().iloc[-1] \
            if len(s_follow[:pred_month - pd.DateOffset(months=1)]) >= w else np.nan

    # DIFF Feature
    for l in range(1, max_self_lag):
        if not pd.isna(rec.get(f'leader_lag{l}')) and not pd.isna(rec.get(f'leader_lag{l+1}')):
            rec[f'leader_diff{l}'] = rec[f'leader_lag{l}'] - rec[f'leader_lag{l+1}']
        if not pd.isna(rec.get(f'follower_lag{l}')) and not pd.isna(rec.get(f'follower_lag{l+1}')):
            rec[f'follower_diff{l}'] = rec[f'follower_lag{l}'] - rec[f'follower_lag{l+1}']

    rec['month_num'] = pred_month.month
    rec['target'] = np.nan

    all_rows.append(rec)

    # 반환
    return pd.DataFrame(all_rows)

In [50]:
all_pred_dfs = []
for leader, follower in tqdm(unique_pair.itertuples(index=False), desc="Processing item pairs"):
    item_feature_df = make_features_by_items(pair_df, leader, follower)

    train_cutoff = pd.Timestamp('2025-07-01') - pd.DateOffset(months=1)
    val_month = pd.Timestamp('2025-07-01')

    train_df_features = item_feature_df[item_feature_df['month'] <= train_cutoff].copy()
    val_df_features = item_feature_df[item_feature_df['month'] == val_month].copy()

    x_cols = [c for c in train_df_features.columns if c not in ['month','leader_item','follower_item','target']]

    train_x = train_df_features[x_cols]
    train_y = train_df_features['target']
    val_x = val_df_features[x_cols]
    val_y = val_df_features['target']

    model = LGBMRegressor(n_estimators=1000, learning_rate=0.05, verbose=-1, random_state=42)
    model.fit(train_x, train_y, eval_set=[(val_x, val_y)], eval_metric='mae')

    pred_df = make_pred_features_by_items(pair_df, leader, follower)
    pred_df['value'] = model.predict(pred_df[x_cols])
    pred_df.loc[pred_df['value'] < 0, 'value'] = 0

    all_pred_dfs.append(pred_df[['leader_item','follower_item','value']])

Processing item pairs: 1425it [06:19,  3.75it/s]


In [51]:
submission = pd.concat(all_pred_dfs, axis=0).sort_values(['leader_item','follower_item']).reset_index(drop=True)
submission.rename(columns={'leader_item': 'leading_item_id', 'follower_item': 'following_item_id'}, inplace=True)
submission.head()

,leading_item_id,following_item_id,value
0,AANGBULD,APQGTRMF,4.269766e+05
1,AANGBULD,DEWLVASR,5.703038e+05
2,AANGBULD,DNMPSKTB,6.338133e+06
3,AANGBULD,EVBVXETX,4.846638e+06
4,AANGBULD,FTSVTTSR,1.153514e+05


In [52]:
submission

,leading_item_id,following_item_id,value
0,AANGBULD,APQGTRMF,4.269766e+05
1,AANGBULD,DEWLVASR,5.703038e+05
2,AANGBULD,DNMPSKTB,6.338133e+06
3,AANGBULD,EVBVXETX,4.846638e+06
4,AANGBULD,FTSVTTSR,1.153514e+05
...,...,...,...
1420,ZXERAXWP,DBWLZWNK,5.004777e+05
1421,ZXERAXWP,FITUEHWN,7.319545e+04
1422,ZXERAXWP,MIRCVAMV,0.000000e+00
1423,ZXERAXWP,UIFPPCLR,1.656951e+05


In [53]:
submission.to_csv('submission4.csv', index=False)

pd.read_csv('submission4.csv')

,leading_item_id,following_item_id,value
0,AANGBULD,APQGTRMF,4.269766e+05
1,AANGBULD,DEWLVASR,5.703038e+05
2,AANGBULD,DNMPSKTB,6.338133e+06
3,AANGBULD,EVBVXETX,4.846638e+06
4,AANGBULD,FTSVTTSR,1.153514e+05
...,...,...,...
1420,ZXERAXWP,DBWLZWNK,5.004777e+05
1421,ZXERAXWP,FITUEHWN,7.319545e+04
1422,ZXERAXWP,MIRCVAMV,0.000000e+00
1423,ZXERAXWP,UIFPPCLR,1.656951e+05
